# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [8]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
# NOMBRE_DATASET      = "LI-Small_Trans.csv"
NOMBRE_DATASET      = "transacciones_sample.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_DATASET}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)
# Filtrar USD antes de samplear
transacciones = transacciones_completas[
    transacciones_completas["Payment Currency"] == "US Dollar"
]

In [9]:
transacciones["Timestamp"]   = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")
transacciones = transacciones.dropna(subset=["Amount Paid", "Payment Format"])

periodo_temprano = transacciones[
    (transacciones["Timestamp"] >= "2022-09-01") &
    (transacciones["Timestamp"] < "2022-09-06")
]
periodo_tardio = transacciones[
    (transacciones["Timestamp"] >= "2022-09-06") &
    (transacciones["Timestamp"] < "2022-09-16")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones")

Período temprano (1/9-5/9): 20189 transacciones
Período tardío  (6/9-15/9): 16833 transacciones


In [10]:
import hashlib

def obtener_id_shard(valor, total_shards):
    valor_norm = str(valor).strip()
    hash_hex = hashlib.md5(valor_norm.encode('utf-8')).hexdigest()
    return (int(hash_hex, 16) % total_shards) + 1

formatos = ["ACH", "Wire", "Cheque", "Cash", "Credit Card", "Reinvestment"]
for f in formatos:
    print(f"{f} -> shard {obtener_id_shard(f, 3)}")

ACH -> shard 3
Wire -> shard 2
Cheque -> shard 3
Cash -> shard 3
Credit Card -> shard 2
Reinvestment -> shard 3


In [11]:
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

                        suma  count      promedio
Payment Format                                   
ACH             8.443794e+08   2372  3.559778e+05
Cash            8.113632e+08   1795  4.520129e+05
Cheque          3.073541e+09   6810  4.513277e+05
Credit Card     1.667260e+07   4865  3.427051e+03
Reinvestment    6.968799e+08   3739  1.863814e+05
Wire            1.822617e+09    608  2.997725e+06


In [12]:
df = periodo_tardio.copy()
df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"])
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
][["Account", "Amount Paid"]].rename(columns={"Account": "From Account"}).reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

Resultados Q3: 9189 transacciones


,From Account,Amount Paid
0,800A74AB0,215.47
1,804C4E2C0,17.64
2,80F68A8A0,27.05
3,80146FB50,200.58
4,80018F9B0,77.58


In [13]:
# Validación del resultado
df_validacion = resultado_query3.copy()
df_validacion = df_validacion.merge(
    periodo_tardio[["Account", "Payment Format", "Timestamp", "Amount Paid"]].rename(columns={"Account": "From Account"}),
    on=["From Account", "Amount Paid"],
    how="left"
)
df_validacion["Promedio Formato"] = df_validacion["Payment Format"].map(stats_por_formato["promedio"])
df_validacion["1% Promedio"] = df_validacion["Promedio Formato"] * 0.01
df_validacion["Es valido"] = df_validacion["Amount Paid"] < df_validacion["1% Promedio"]

print(f"Filas totales: {len(df_validacion)}")
print(f"Filas válidas: {df_validacion['Es valido'].sum()}")
print(f"Filas inválidas: {(~df_validacion['Es valido']).sum()}")
df_validacion.head(10)

Filas totales: 9735
Filas válidas: 9733
Filas inválidas: 2


,From Account,Amount Paid,Payment Format,Timestamp,Promedio Formato,1% Promedio,Es valido
0,800A74AB0,215.47,Cash,2022-09-09 00:38:00,4.520129e+05,4520.129179,True
1,804C4E2C0,17.64,Wire,2022-09-06 17:39:00,2.997725e+06,29977.250218,True
2,80F68A8A0,27.05,Credit Card,2022-09-10 00:15:00,3.427051e+03,34.270507,True
3,80146FB50,200.58,Wire,2022-09-08 12:41:00,2.997725e+06,29977.250218,True
4,80018F9B0,77.58,ACH,2022-09-09 11:27:00,3.559778e+05,3559.778249,True
5,816330270,2.67,Credit Card,2022-09-08 00:26:00,3.427051e+03,34.270507,True
6,806E323F0,235.23,Cheque,2022-09-09 23:28:00,4.513277e+05,4513.276772,True
7,809A726F0,4393.54,Cheque,2022-09-09 19:47:00,4.513277e+05,4513.276772,True
8,80DDB36D0,1002.99,ACH,2022-09-07 11:53:00,3.559778e+05,3559.778249,True
9,800057A30,58.45,ACH,2022-09-10 20:27:00,3.559778e+05,3559.778249,True


In [14]:
print(f"Período temprano total: {len(periodo_temprano)}")
print(f"Período tardío total: {len(periodo_tardio)}")
print("\nTemprano por formato:")
print(periodo_temprano.groupby("Payment Format")["Amount Paid"].count())
print("\nTardío por formato:")
print(periodo_tardio.groupby("Payment Format")["Amount Paid"].count())

Período temprano total: 20189
Período tardío total: 16833

Temprano por formato:
Payment Format
ACH             2372
Cash            1795
Cheque          6810
Credit Card     4865
Reinvestment    3739
Wire             608
Name: Amount Paid, dtype: int64

Tardío por formato:
Payment Format
ACH            2228
Cash           1803
Cheque         7172
Credit Card    5006
Wire            624
Name: Amount Paid, dtype: int64
